# 1D J1-J2=0.0-J3=0.5: LorentzRNN

This is part of the work https://arxiv.org/abs/2604.24337. 

In [1]:
import os
import sys
sys.path.append('../utility_lorentz')
from j1j2j3_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
E_exact =-53.99140745384424
units = 80
syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.0
J3 = 0.5
lr1=5e-3
lr2=5e-3

nsteps = 1201
var_tol = 20.0

In [3]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

# LorentzRNN

With double clamps (on 'state' and 'new_h') in the forward pass, the performance of LorentzRNN dramatically improved compared to when there was a single clamp (on 'state' only).

In [4]:
#DOUBLE CLAMP INSIDE FORWARD PASS
spatial_clamp=4.0
fname=f'1d_J1J2J3_results_Lorentz_N={syssize}/LorentzRNN_sc={spatial_clamp}/J2={J2}_J3={J3}'
wf2=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,
                        spatial_clamp=spatial_clamp, seed=111)
for name, param in wf2.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params ={sum(p.numel() for p in wf2.model.parameters())}')

t0=time.time()
mE, vE = run_J1J2J3(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=5e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([80, 80])
Layer: rnn.U | Size: torch.Size([80, 2])
Layer: rnn.b | Size: torch.Size([80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])
Total params =6965


/Users/hl/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/Users/hl/anaconda3/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:1052: ComplexWarning: Casting complex values to real discards the imaginary part
  current = float(metrics)


step: 0, loss: -1.20361, mean energy: 36.86465-0.00062j|varE: 0.37611| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.05
max_h0 = 1.0000944137573242 | max_h_spatial_norm = 0.013743148185312748| max_h_violation = 5.960464477539063e-08
Best model saved at epoch 1 with best E=36.83099-0.05484j, varE=0.59668
Best model saved at epoch 3 with best E=35.06470-0.02208j, varE=6.88994
step: 10, loss: 53.09570, mean energy: -31.91685+0.14820j|varE: 50.51824| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.0490000000000002
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 5.7220458984375e-06
step: 20, loss: -13.89050, mean energy: -31.72037+0.13990j|varE: 48.39384| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.048
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 5.7220458984375e-06
step: 30, loss: 6.96375, mean energy: -33.91344-0.06944j|varE: 71.65189| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.0470000000000002
max_h0 = 4.123096942901611 | max_h_spatial_

step: 260, loss: 30.24829, mean energy: -52.29200+0.21615j|varE: 6.11951| Hyp LR: 2.50e-03| LR: 2.50e-03| tau=1.024
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 5.7220458984375e-06
step: 270, loss: 43.12256, mean energy: -51.80836-0.03773j|varE: 6.52482| Hyp LR: 2.50e-03| LR: 2.50e-03| tau=1.0230000000000001
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 7.62939453125e-06
step: 280, loss: 23.27319, mean energy: -52.17124-0.19974j|varE: 4.27702| Hyp LR: 2.50e-03| LR: 2.50e-03| tau=1.022
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 7.62939453125e-06
Best model saved at epoch 287 with best E=-53.08542+0.01332j, varE=2.81529
step: 290, loss: -7.10474, mean energy: -52.64594-0.01948j|varE: 3.72928| Hyp LR: 1.25e-03| LR: 1.25e-03| tau=1.0210000000000001
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 7.62939453125e-06
Best model sa

step: 590, loss: -9.37964, mean energy: -53.67715+0.00508j|varE: 1.79640| Hyp LR: 7.81e-05| LR: 7.81e-05| tau=1.0
max_h0 = 4.123096466064453 | max_h_spatial_norm = 3.999990701675415| max_h_violation = 7.62939453125e-06
step: 600, loss: -0.58960, mean energy: -53.81477+0.02254j|varE: 1.37309| Hyp LR: 7.81e-05| LR: 7.81e-05| tau=1.0
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 7.62939453125e-06
step: 610, loss: -2.16553, mean energy: -53.73730+0.00950j|varE: 1.45208| Hyp LR: 7.81e-05| LR: 7.81e-05| tau=1.0
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 7.62939453125e-06
step: 620, loss: -2.30542, mean energy: -53.64833+0.00999j|varE: 0.94952| Hyp LR: 3.91e-05| LR: 3.91e-05| tau=1.0
max_h0 = 4.123096942901611 | max_h_spatial_norm = 3.999990940093994| max_h_violation = 7.62939453125e-06
Best model saved at epoch 622 with best E=-54.16166+0.06730j, varE=4.92898
step: 630, loss: -3.62012, mean energy: -53.38904

## Single clamp in forward pass

In [5]:
spatial_clamp=2.0
fname=f'1d_J1J2J3_results_Lorentz_N={syssize}/LorentzRNN_sc={spatial_clamp}/J2={J2}_J3={J3}'
#spatial_clamp=3.0: stuck at -37.0
wf2=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=111)
for name, param in wf2.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params ={sum(p.numel() for p in wf2.model.parameters())}')

t0=time.time()
mE, vE = run_J1J2J3(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=5e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([80, 80])
Layer: rnn.U | Size: torch.Size([80, 2])
Layer: rnn.b | Size: torch.Size([80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])
Total params =6965
step: 0, loss: -1.20508, mean energy: 36.86506-0.00013j|varE: 0.37717| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.05
Best model saved at epoch 3 with best E=35.81402-0.00130j, varE=4.75673
Best model saved at epoch 4 with best E=33.18544-0.21342j, varE=15.32012
step: 10, loss: 125.58466, mean energy: -18.39806+0.60383j|varE: 67.04112| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.0490000000000002
step: 20, loss: -0.78581, mean energy: -35.79800+0.18827j|varE: 2.18713| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.048
Best model saved at epoch 22 with best E=-36.18454-0.00444j, varE=2.01036
Best model saved at epoch 23 with best E=-36.27412+0.07

In [12]:
spatial_clamp=4.0
fname=f'1d_J1J2J3_results_Lorentz_N={syssize}/LorentzRNN_sc={spatial_clamp}/J2={J2}_J3={J3}'
#spatial_clamp=3.0: stuck at -37.0
wf2=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzRNN', units=units,spatial_clamp=spatial_clamp, seed=111)
for name, param in wf2.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params ={sum(p.numel() for p in wf2.model.parameters())}')

t0=time.time()
mE, vE = run_J1J2J3(wf2, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=lr1, lr2=lr2, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

Layer: rnn.W | Size: torch.Size([80, 80])
Layer: rnn.U | Size: torch.Size([80, 2])
Layer: rnn.b | Size: torch.Size([80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Layer: manifold.k | Size: torch.Size([])
Total params =6965
step: 0, loss: -1.20581, mean energy: 36.86463+0.00172j|varE: 0.37834| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.05
Best model saved at epoch 1 with best E=36.82828+0.05849j, varE=0.79898
Best model saved at epoch 2 with best E=36.32803-0.17124j, varE=2.62921
step: 10, loss: 5.50254, mean energy: 1.77089-0.24805j|varE: 60.66974| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.0490000000000002
step: 20, loss: -136.56079, mean energy: -22.80224+0.50782j|varE: 66.35145| Hyp LR: 5.00e-03| LR: 5.00e-03| tau=1.048
Best model saved at epoch 30 with best E=-35.28905-0.04451j, varE=6.01611
step: 30, loss: -5.79938, mean energy: -35.28905-0.04451j